In [1]:
pip install pymc

Note: you may need to restart the kernel to use updated packages.


In [20]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import pymc as pm

In [4]:
da=load_breast_cancer()

In [5]:
X,y=da.data,da.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
mos=[LogisticRegression(max_iter=1000),DecisionTreeClassifier(),SVC(probability=True)]

In [15]:
preds=[]
for mo in mos:
    mo.fit(X_train,y_train)
    pred=mo.predict(X_test)
    preds.append(pred)

In [17]:
preds=np.array(preds)

In [18]:
U=(preds==y_test).astype(int)

In [19]:
U

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [21]:
nm,nq=U.shape

In [22]:
with pm.Model() as irt_model:
    t=pm.Normal('theta',mu=0,sigma=1,shape=nm)
    b=pm.Normal('b',mu=0,sigma=1,shape=nq)
    a=pm.LogNormal('a',mu=0,sigma=1,shape=nq)
    logs=a*(t[:,None]-b[None,:])
    p=pm.math.sigmoid(logs)
    obs=pm.Bernoulli('obs',p=p,observed=U)
    tr=pm.sample(1000,tune=1000,chains=2)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [theta, b, a]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 4 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [23]:
theta_mean = tr.posterior["theta"].mean(dim=["chain", "draw"])
print("Model abilities:", theta_mean.values)

Model abilities: [3.59348933 2.40409542 3.44442462]


In [25]:
theta_std = tr.posterior["theta"].std(dim=["chain", "draw"])
print("Uncertainty:", theta_std.values)

Uncertainty: [0.59807681 0.40580689 0.53996003]


In [27]:
samples = tr.posterior["theta"].values

prob_model0_better_1 = (samples[:,:,0] > samples[:,:,2]).mean()
print(prob_model0_better_1)

0.574
